# GraviLens v2 — Colab driver

One notebook, no Drive, no local setup. It needs two things:

1. **A GPU runtime** (Runtime → Change runtime type → T4 GPU). Data generation is
   CPU-bound and runs in the background, so the GPU is free while it works.
2. **The package source** — either a published GitHub repo (set `REPO` below) or a
   source zip you upload from the browser. Create the zip on your machine with:
   `git archive --format=zip --prefix=gravilens-pkg/ -o gravilens-src.zip v0.1.0`

Then: dataset (upload or regenerate) → 2-epoch smoke → 21-run grid → audit → table.
No result is claimed anywhere in this repository until the grid has been run.

In [ ]:
import os, shutil, subprocess, sys, zipfile

def run(cmd, **kw):
    """Run a command with THIS interpreter; `python` on PATH may differ."""
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    return subprocess.run(cmd, check=True, **kw)

print('interpreter:', sys.executable)
print('python    :', sys.version.split()[0])

In [ ]:
# --- get the package source ----------------------------------------------
REPO = ''            # e.g. 'https://github.com/<handle>/gravilens'; '' = upload instead
TAG = 'v0.1.0'
SRC = '/content/gravilens'

if not os.path.isdir(os.path.join(SRC, 'gravilens')):
    if REPO:
        try:
            run(['git', 'clone', REPO, SRC])
            run(['git', '-C', SRC, 'fetch', '--tags'])
            run(['git', '-C', SRC, 'checkout', TAG])
        except Exception as exc:
            print('clone failed:', type(exc).__name__, exc)
    if not os.path.isdir(os.path.join(SRC, 'gravilens')):
        print('Upload gravilens-src.zip (see the markdown above).')
        from google.colab import files
        for name, payload in files.upload().items():
            with open('/content/' + name, 'wb') as handle:
                handle.write(payload)
            with zipfile.ZipFile('/content/' + name) as archive:
                archive.extractall('/content/')
        if os.path.isdir('/content/gravilens-pkg'):
            shutil.move('/content/gravilens-pkg', SRC)
else:
    print('source already at', SRC)

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', SRC])
import gravilens
print('gravilens', gravilens.__file__)

In [ ]:
import torch
print('torch', torch.__version__)
print('gpu  ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## Data

**Path A (fast): upload the dataset.** Prepare it on your machine with
`tar -cf gravilens-v2.tar -C data gravilens-v2` (~300 MB) and pick it with the file
uploader below. The fingerprint is content-based, so any copy of the same data
works.

**Path B (default): regenerate.** ~4 CPU-hours for the train split; `val` and `test`
finish much sooner. The pixels are bit-identical to an uploaded copy of the same
version, because rendering is deterministic per triplet seed.

In [ ]:
# Path A: use an uploaded dataset archive. Set UPLOAD=True to pick the file.
import tarfile

UPLOAD = False
if UPLOAD:
    from google.colab import files
    for name, payload in files.upload().items():
        path = '/content/' + name
        with open(path, 'wb') as handle:
            handle.write(payload)
        if name.endswith('.zip'):
            with zipfile.ZipFile(path) as archive:
                archive.extractall('/content/')
        else:
            with tarfile.open(path) as archive:
                archive.extractall('/content/')
    print('extracted:', os.path.isdir('/content/gravilens-v2'))

In [ ]:
# Path B: regenerate from scratch, one process per split, all using this
# interpreter. ~4 CPU-hours for the train split.
REGENERATE = True
if REGENERATE:
    procs = {}
    for split in ('train', 'val', 'test'):
        log = open(f'/content/{split}_gen.log', 'w')
        procs[split] = (subprocess.Popen(
            [sys.executable, '-m', 'gravilens', 'generate',
             '--config', 'configs/data.yaml', '--splits', split],
            stdout=log, stderr=subprocess.STDOUT, cwd=SRC), log)
    for split, (proc, log) in procs.items():
        proc.wait()
        log.close()
    for split in procs:
        lines = open(f'/content/{split}_gen.log').read().strip().splitlines()
        print(split, ':', lines[-1] if lines else '(no output)')

In [ ]:
# One root for every later command, plus a completeness check: a grid trained
# on a partial dataset is rejected by `audit`.
from gravilens.dataset import load_split

UPLOADED = '/content/gravilens-v2'
REGENERATED = os.path.join(SRC, 'data', 'gravilens-v2')
DATA_ROOT = UPLOADED if os.path.isdir(UPLOADED) else REGENERATED
print('DATA_ROOT =', DATA_ROOT)

run([sys.executable, '-m', 'gravilens', 'dataset',
     '--data-root', DATA_ROOT, '--split', 'all'])

train = load_split(DATA_ROOT, 'train')
print(f'train images: {len(train):,} (the audited grid needs 6,999)')
if len(train) != 6999:
    print('WARNING: incomplete dataset. A grid trained on it is rejected by')
    print('         `audit` (train_size != 6999). Regenerate or upload the')
    print('         complete dataset before running the grid.')

## Smoke test (2 epochs)

Checks the full train/evaluate path before spending GPU time. The artifacts go to
`runs-smoke/`, never to `runs/`, so they cannot be mistaken for grid results.

In [ ]:
run([sys.executable, '-m', 'gravilens', 'train',
     '--config', 'configs/ablation.yaml', '--data-root', DATA_ROOT,
     '--model', 'gravilens', '--variant', 'full', '--seed', '42',
     '--epochs', '2', '--out', 'runs-smoke/gravilens_full_seed42'], cwd=SRC)
run([sys.executable, '-m', 'gravilens', 'evaluate',
     '--data-root', DATA_ROOT,
     '--run', 'runs-smoke/gravilens_full_seed42'], cwd=SRC)

## Full ablation (21 runs)

4 variants × 3 seeds + 3 parameter-matched baselines × 3 seeds, in the
pre-registered order: the reviewer-critical variants first, the baselines second,
`dist` last. `ablate` reuses a run only when it used the same recipe and the same
dataset. `audit` exits non-zero until all 21 runs exist and every run used the
pre-registered recipe; `figures` writes the table from the artifacts only.

Runtimes are ephemeral: rerun a cell to continue — finished runs are skipped.

In [ ]:
# Chunk 1: the reviewer-critical variants (3 variants x 3 seeds).
run([sys.executable, '-m', 'gravilens', 'ablate',
     '--config', 'configs/ablation.yaml', '--data-root', DATA_ROOT,
     '--variants', 'full', 'no_sis', 'no_sup'], cwd=SRC)

In [ ]:
# Preview from the runs that exist so far. This is NOT the result: the final
# table comes from `figures` after all 21 runs are audited.
import glob, json

rows = []
for path in sorted(glob.glob(os.path.join(SRC, 'runs', '*', 'metrics.json'))):
    m = json.load(open(path))
    rows.append((m['run'], m['test']['macro_f1'], m['test']['macro_auc']))
for name, f1, auc in rows:
    print(f'{name:34s} macro-F1 {f1:.3f}  AUC {auc:.3f}')

def mean_macro_f1(variant):
    vals = [f1 for name, f1, _ in rows if name.startswith(f'gravilens_{variant}_')]
    return sum(vals) / len(vals) if vals else None

full, no_sis = mean_macro_f1('full'), mean_macro_f1('no_sis')
if full is not None and no_sis is not None:
    print(f'partial: {len(rows)}/21 runs | full - no_sis = {full - no_sis:+.3f} macro-F1')

In [ ]:
# Chunk 2: the three parameter-matched baselines.
run([sys.executable, '-m', 'gravilens', 'ablate',
     '--config', 'configs/ablation.yaml', '--data-root', DATA_ROOT,
     '--baselines', 'vit', 'cait', 'vitsd'], cwd=SRC)

In [ ]:
# Chunk 3: the `dist` variant, then the audit and the table.
run([sys.executable, '-m', 'gravilens', 'ablate',
     '--config', 'configs/ablation.yaml', '--data-root', DATA_ROOT,
     '--variants', 'dist'], cwd=SRC)
run([sys.executable, '-m', 'gravilens', 'audit'], cwd=SRC)
run([sys.executable, '-m', 'gravilens', 'figures'], cwd=SRC)

## Save artifacts

Colab runtimes are ephemeral. The cell below zips `runs/` and downloads it.

In [ ]:
run(['zip', '-qr', '/content/runs.zip', 'runs'], cwd=SRC)
from google.colab import files
files.download('/content/runs.zip')